# Partie 3 — Pollution urbaine : OpenAQ

Objectif : comparer les concentrations, étudier leur variation journalière, repérer le polluant dominant et détecter des anomalies. Les villes du projet sont étudiées séparément ; la couverture réelle des capteurs doit être vérifiée avant interprétation.

Préparation : `python -m src.iot_openaq.history --start 2021-01-01 --end 2025-08-31`, puis `python -m src.iot_openaq.process`. Une clé `OPENAQ_API_KEY` est nécessaire pour la collecte. L'absence de capteur dans une ville n'est pas une concentration nulle.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from src.iot_openaq.config import DATA_DIR
from src.analysis_cross.prepare import pollution_tables
path = DATA_DIR / 'processed/openaq/pollution_clean.parquet'
if not path.exists():
    raise FileNotFoundError('Données OpenAQ réelles manquantes : récupérer les fichiers du groupe ou lancer history puis process avec une clé API.')
df = pd.read_parquet(path)
hourly, daily = pollution_tables(df)
if hourly.empty:
    raise ValueError('Aucune mesure valide dans les villes du projet. Vérifier coverage_locations.json et les dates demandées.')
FIG = ROOT / 'reports/figures/partie3'
FIG.mkdir(parents=True, exist_ok=True)
OUT = ROOT / 'reports/partie3'
OUT.mkdir(parents=True, exist_ok=True)
display(hourly.groupby(['city','pollutant','unit']).agg(n=('pollution','count'), debut=('heure_utc','min'), fin=('heure_utc','max')))

def figure_table(table, name, title):
    if table.empty:
        print(title + ' : données insuffisantes, aucune figure générée.')
        return
    table.to_csv(OUT / (name + '.csv'))
    ax = table.plot(kind='bar', figsize=(12,5))
    ax.set_title(title)
    ax.set_ylabel('Concentration dans l’unité indiquée')
    plt.tight_layout()
    plt.savefig(FIG / (name + '.png'), dpi=140, bbox_inches='tight')
    plt.show()

## 1. Quelles villes présentent les concentrations les plus élevées ?

Les comparaisons sont séparées par polluant et unité. Chaque concentration horaire est une moyenne de capteurs, pas un nombre de mesures. Les périodes et nombres de capteurs peuvent différer : le classement décrit uniquement l'échantillon disponible.

In [ ]:
concentrations = hourly.groupby(['pollutant','unit','city']).pollution.agg(['mean','median','count'])
display(concentrations)
concentrations.to_csv(OUT / 'concentrations_villes.csv')
for (pollutant, unit), g in hourly.groupby(['pollutant','unit']):
    figure_table(g.groupby('city').pollution.mean().to_frame(pollutant),
                 f'q1_{pollutant}_{len(g)}', f'{pollutant} — concentration moyenne ({unit})')

## 2. Comment la pollution varie-t-elle dans la journée ?

Les heures sont converties en Europe/Paris pour distinguer les rythmes locaux. Les effectifs sont affichés ; un profil établi sur un seul jour ne démontre pas un cycle reproductible. Les jours civils ne comptant pas au moins 18 heures par capteur sont exclus du tableau journalier.

In [ ]:
local = hourly.copy()
local['heure_locale'] = local.heure_utc.dt.tz_convert('Europe/Paris').dt.hour
local['date_locale'] = local.heure_utc.dt.tz_convert('Europe/Paris').dt.date
profil = local.groupby(['city','pollutant','unit','heure_locale']).agg(concentration=('pollution','mean'), n=('pollution','count'), jours=('date_locale','nunique'))
display(profil)
profil.to_csv(OUT / 'q2_profils_journaliers.csv')
for (city, pollutant, unit), g in local.groupby(['city','pollutant','unit']):
    figure_table(g.groupby('heure_locale').pollution.mean().to_frame(pollutant),
                 f'q2_{city}_{pollutant}', f'{city} : profil horaire {pollutant} ({unit})')
display(daily.head(30))

## 3. Quel polluant a la concentration dominante ?

Le nombre de mesures décrit la couverture, pas la dominance. Ici, on compare les **concentrations massiques moyennes aux mêmes heures**, uniquement entre polluants exprimés en µg/m³ dans une même ville. Les unités ppb/ppm ne sont pas converties implicitement. Une concentration massique supérieure ne signifie pas une toxicité supérieure ; aucun classement sanitaire n'est déduit.

In [ ]:
mass = hourly[hourly.unit.isin(['µg/m³','µg/m3'])]
rows = []
for city, g in mass.groupby('city'):
    pivot = g.pivot_table(index='heure_utc', columns='pollutant', values='pollution')
    common = pivot.dropna()
    if len(pivot.columns) < 2 or common.empty:
        print(city, ': comparaison impossible, moins de deux polluants sur des heures communes.')
        continue
    means = common.mean().sort_values(ascending=False)
    rows.append(dict(city=city, polluant_dominant=means.index[0], concentration=means.iloc[0], unite='µg/m³', heures_communes=len(common)))
    figure_table(means.to_frame('concentration'), f'q3_{city}', f'{city} : concentrations sur {len(common)} heures communes (µg/m³)')
dominants = pd.DataFrame(rows)
display(dominants)
if not dominants.empty: dominants.to_csv(OUT / 'q3_dominants.csv', index=False)

## 4. Où observe-t-on des concentrations atypiques ?

Le seuil exploratoire est Q3 + 1,5 × IQR, par ville/polluant/unité, avec au moins 30 heures sur sept dates. Il ne s'agit pas d'un seuil réglementaire ni d'une preuve d'incident. Les changements de capteurs ou de météo peuvent expliquer ces valeurs.

In [ ]:
anomalies=[]
for (city, pollutant, unit), g in local.groupby(['city','pollutant','unit']):
    if len(g) < 30 or g.date_locale.nunique() < 7:
        continue
    q1,q3 = g.pollution.quantile([.25,.75])
    if q3 <= q1: continue
    threshold=q3+1.5*(q3-q1)
    peaks=g[g.pollution > threshold].assign(seuil=threshold)
    anomalies.append(peaks)
    ax=g.plot(x='heure_utc',y='pollution',figsize=(12,4),title=f'{city} — {pollutant} ({unit})')
    ax.axhline(threshold,color='red',label='seuil exploratoire')
    ax.legend(); plt.tight_layout()
    plt.savefig(FIG / f'q4_{city}_{pollutant}.png',dpi=140); plt.show()
if anomalies:
    peaks=pd.concat(anomalies,ignore_index=True)
    display(peaks)
    peaks.to_csv(OUT / 'q4_anomalies.csv',index=False)
else:
    print('Couverture ou variabilité insuffisante : détection non calculable.')

## Interprétation finale

Les tableaux permettent d'identifier les concentrations moyennes observées par ville, les profils horaires réellement couverts et les concentrations atypiques. Le polluant dominant est comparé en concentration sur un échantillon temporel commun, pas en fréquence de mesure.

Les conclusions restent propres aux capteurs et périodes présents. L'évolution des capteurs, l'implantation près du trafic et la couverture inégale limitent les comparaisons. Des données absentes ou insuffisantes ne permettent ni de déclarer une ville peu polluée ni d'affirmer l'absence de pics. Les interprétations chiffrées doivent être rédigées après collecte et lecture des tableaux réels.

Sources : [OpenAQ locations](https://docs.openaq.org/resources/locations), [OpenAQ sensors](https://docs.openaq.org/resources/sensors). Les concentrations ne constituent pas à elles seules un classement sanitaire.